# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShubhamSnSharma/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
# Imports
import duckdb
import pandas as pd

from getpass import getpass

# Authenticate with Hugging Face
token = getpass("Enter your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

# Dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"

Enter your Hugging Face READ token: ··········


In [5]:
feature_df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    COALESCE(client_has_gsc, FALSE) AS client_has_gsc,
    COALESCE(client_has_ga4, FALSE) AS client_has_ga4,
    COALESCE(gsc_data_available, FALSE) AS gsc_data_available,
    COALESCE(ga4_data_available, FALSE) AS ga4_data_available,

    COALESCE(gsc_impressions, 0) AS gsc_impressions,
    COALESCE(gsc_clicks, 0) AS gsc_clicks,
    gsc_avg_position,

    COALESCE(ga4_pageviews, 0) AS ga4_pageviews,
    COALESCE(ga4_engaged_sessions, 0) AS ga4_engaged_sessions

FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)

WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
 feature_df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_pageviews",
    ]
].describe(percentiles=[0.5,0.75,0.9,0.95,0.99]).T

,count,mean,std,min,50%,75%,90%,95%,99%,max
gsc_impressions,9841378.0,28.518119,155.926569,0.0,0.0,6.0,54.0,135.00,509.00,40084.0
gsc_avg_position,3611061.0,15.826651,19.856034,0.0,7.5,20.2,43.0,62.75,88.75,498.0
ga4_pageviews,9841378.0,0.150879,1.787007,0.0,0.0,0.0,0.0,0.00,4.00,875.0


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

---

## My baseline rule

### Goal

Prioritize pages that already receive search visibility and user activity but still have room for improvement through a content refresh.

### Rule

A page receives a higher baseline score when it satisfies more of the following conditions:

- receives at least **10 GSC impressions**
- ranks **outside the top 10 search positions**
- receives at least **one GA4 pageview**

These thresholds were chosen after reviewing the Week 4 signal distributions and bucket analyses. All three signals are available before prediction and do not introduce target leakage.

### Action label

`REFRESH`

### Reason codes

| Reason code | Meaning |
|-------------|---------|
| `HIGH_VISIBILITY` | The page receives at least 10 GSC impressions. |
| `MID_RANKING` | The page ranks below the top 10 search positions. |
| `HAS_TRAFFIC` | The page receives at least 1 GA4 pageview. |

In [7]:
# Make a working copy
baseline_df = feature_df.copy()

# ----------------------------------------------------
# Baseline score
# One point for each rule that is satisfied
# ----------------------------------------------------

baseline_df["baseline_score"] = 0

# Rule 1: already receives meaningful search visibility
baseline_df.loc[
    baseline_df["gsc_impressions"] >= 10,
    "baseline_score"
] += 1

# Rule 2: ranks outside the Top 10 (room for improvement)
baseline_df.loc[
    baseline_df["gsc_avg_position"] > 10,
    "baseline_score"
] += 1

# Rule 3: already receives some user traffic
baseline_df.loc[
    baseline_df["ga4_pageviews"] >= 1,
    "baseline_score"
] += 1


# ----------------------------------------------------
# Reason codes
# Store every rule that contributed to the score
# ----------------------------------------------------

def get_reason(row):
    reasons = []

    if row["gsc_impressions"] >= 10:
        reasons.append("HIGH_VISIBILITY")

    if row["gsc_avg_position"] > 10:
        reasons.append("MID_RANKING")

    if row["ga4_pageviews"] >= 1:
        reasons.append("HAS_TRAFFIC")

    if not reasons:
        return "LOW_PRIORITY"

    return "; ".join(sorted(reasons))


baseline_df["reason_code"] = baseline_df.apply(get_reason, axis=1)


# ----------------------------------------------------
# Action label
# ----------------------------------------------------

baseline_df["action"] = "REFRESH"


# ----------------------------------------------------
# Preview highest-scoring pages
# ----------------------------------------------------

preview = baseline_df.sort_values(
    by=[
        "baseline_score",
        "gsc_impressions",
        "ga4_pageviews",
        "gsc_avg_position",
    ],
    ascending=[False, False, False, True]
)

display(
    preview[
        [
            "content_hash_id",
            "gsc_impressions",
            "gsc_avg_position",
            "ga4_pageviews",
            "baseline_score",
            "reason_code",
            "action",
        ]
    ].head(10)
)

print("Baseline score distribution:")
display(
    baseline_df["baseline_score"]
    .value_counts()
    .sort_index()
)

print("Reason code distribution:")
display(
    baseline_df["reason_code"]
    .value_counts()
)

,content_hash_id,gsc_impressions,gsc_avg_position,ga4_pageviews,baseline_score,reason_code,action
8860963,content_66288edeb93b7c4f,24577,10.794239,459,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH
9650754,content_66288edeb93b7c4f,23542,11.112140,149,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH
2774228,content_e8a52cf3d5988c07,15394,17.296804,34,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH
9164647,content_e6df0936699f5b8f,14682,25.035826,172,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH
3615213,content_e8a52cf3d5988c07,14138,16.244306,50,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH
3808137,content_e8a52cf3d5988c07,13910,16.680446,28,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH
2948133,content_e8a52cf3d5988c07,13060,16.536753,47,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH
1690012,content_e8a52cf3d5988c07,11347,16.566053,71,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH
346656,content_e8a52cf3d5988c07,11019,15.402033,114,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH
9163168,content_74de5f247659e956,10907,18.572293,62,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH


Baseline score distribution:


,count
baseline_score,
0,6998439
1,1858856
2,822682
3,161401


Reason code distribution:


,count
reason_code,
LOW_PRIORITY,6998439
HIGH_VISIBILITY,1177626
HIGH_VISIBILITY; MID_RANKING,642477
MID_RANKING,609519
HAS_TRAFFIC; HIGH_VISIBILITY,166025
HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,161401
HAS_TRAFFIC,71711
HAS_TRAFFIC; MID_RANKING,14180


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

---

## Ranked baseline queue

Every page is ranked using the baseline score from Section 1.

When multiple pages receive the same baseline score, ties are broken using:

1. Higher GSC impressions
2. Higher GA4 pageviews
3. Better (lower) average search position

The ranked queue is then written to:

`work/outputs/baseline_action_score.csv`

In [8]:
from pathlib import Path

# Rank pages
ranked_df = (
    baseline_df
    .sort_values(
        by=[
            "baseline_score",
            "gsc_impressions",
            "ga4_pageviews",
            "gsc_avg_position",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

# Ranking number
ranked_df["rank"] = ranked_df.index + 1

# Columns to export
output_df = ranked_df[
    [
        "rank",
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_pageviews",
    ]
]

# Create output directory
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

# Save CSV
output_df.to_csv(output_path, index=False)

print(f"Saved {len(output_df):,} rows")
print(f"Output file: {output_path}")

output_df.head(20)

Saved 9,841,378 rows
Output file: work/outputs/baseline_action_score.csv


,rank,content_hash_id,baseline_score,reason_code,action,gsc_impressions,gsc_avg_position,ga4_pageviews
0,1,content_66288edeb93b7c4f,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH,24577,10.794239,459
1,2,content_66288edeb93b7c4f,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH,23542,11.112140,149
2,3,content_e8a52cf3d5988c07,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH,15394,17.296804,34
3,4,content_e6df0936699f5b8f,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH,14682,25.035826,172
4,5,content_e8a52cf3d5988c07,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH,14138,16.244306,50
5,6,content_e8a52cf3d5988c07,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH,13910,16.680446,28
6,7,content_e8a52cf3d5988c07,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH,13060,16.536753,47
7,8,content_e8a52cf3d5988c07,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH,11347,16.566053,71
8,9,content_e8a52cf3d5988c07,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH,11019,15.402033,114
9,10,content_74de5f247659e956,3,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,REFRESH,10907,18.572293,62


## Top-20 review

The baseline rule ranked pages using only three transparent signals:

- search visibility (`gsc_impressions >= 10`)
- ranking opportunity (`gsc_avg_position > 10`)
- existing user traffic (`ga4_pageviews >= 1`)

The highest-ranked pages satisfy all three rules. These pages already receive meaningful visibility and traffic while still ranking outside the top 10 search positions, making them reasonable candidates for a content refresh.

Because this is a rule-based baseline, the ranking is intended for prioritization rather than final decision-making.

In [9]:
top20_review = (
    baseline_df
    .sort_values(
        ["baseline_score", "gsc_impressions"],
        ascending=[False, False]
    )
    .head(20)
    .copy()
)


def confidence_note(row):
    if row["gsc_impressions"] >= 1000:
        return "High confidence. Strong search visibility supports this recommendation."
    return "Medium confidence. Meets the baseline rules but should be reviewed manually."


def what_would_make_it_wrong(row):
    return (
        "The page may already be fully optimized, seasonal, or limited by factors "
        "outside the content, such as backlinks or search intent."
    )


top20_review["confidence_note"] = top20_review.apply(
    confidence_note,
    axis=1
)

top20_review["what_would_make_it_wrong"] = top20_review.apply(
    what_would_make_it_wrong,
    axis=1
)

display(
    top20_review[
        [
            "content_hash_id",
            "baseline_score",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong",
        ]
    ]
)

,content_hash_id,baseline_score,action,reason_code,confidence_note,what_would_make_it_wrong
8860963,content_66288edeb93b7c4f,3,REFRESH,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,High confidence. Strong search visibility supp...,"The page may already be fully optimized, seaso..."
9650754,content_66288edeb93b7c4f,3,REFRESH,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,High confidence. Strong search visibility supp...,"The page may already be fully optimized, seaso..."
2774228,content_e8a52cf3d5988c07,3,REFRESH,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,High confidence. Strong search visibility supp...,"The page may already be fully optimized, seaso..."
9164647,content_e6df0936699f5b8f,3,REFRESH,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,High confidence. Strong search visibility supp...,"The page may already be fully optimized, seaso..."
3615213,content_e8a52cf3d5988c07,3,REFRESH,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,High confidence. Strong search visibility supp...,"The page may already be fully optimized, seaso..."
3808137,content_e8a52cf3d5988c07,3,REFRESH,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,High confidence. Strong search visibility supp...,"The page may already be fully optimized, seaso..."
2948133,content_e8a52cf3d5988c07,3,REFRESH,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,High confidence. Strong search visibility supp...,"The page may already be fully optimized, seaso..."
1690012,content_e8a52cf3d5988c07,3,REFRESH,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,High confidence. Strong search visibility supp...,"The page may already be fully optimized, seaso..."
346656,content_e8a52cf3d5988c07,3,REFRESH,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,High confidence. Strong search visibility supp...,"The page may already be fully optimized, seaso..."
9163168,content_74de5f247659e956,3,REFRESH,HAS_TRAFFIC; HIGH_VISIBILITY; MID_RANKING,High confidence. Strong search visibility supp...,"The page may already be fully optimized, seaso..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

---

### Possible weak picks

The baseline rule only considers three transparent signals. Some selected pages may still be poor refresh candidates because the rule cannot measure:

- search intent changes
- seasonal traffic
- backlink quality
- content quality
- business priorities

For example, a page may receive impressions and traffic but already be fully optimized, making a refresh unlikely to produce meaningful gains.

### Leakage check

No future information was used when constructing the baseline rule.

The score only uses features available before prediction:

- GSC impressions
- GSC average position
- GA4 pageviews

No product outcome flags, future labels, or post-refresh information were included, so the baseline does not introduce target leakage.

In [10]:
leakage_review = {
    "Uses only pre-prediction signals": True,
    "Uses future labels": False,
    "Uses product outcome flags": False,
    "Uses post-refresh metrics": False,
    "Risk of target leakage": "Low"
}

display(pd.DataFrame([leakage_review]))

,Uses only pre-prediction signals,Uses future labels,Uses product outcome flags,Uses post-refresh metrics,Risk of target leakage
0,True,False,False,False,Low


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.